# AlphaLens SHAP and Error Analysis

This notebook explains the selected XGBoost model on its untouched test fold. It verifies SHAP additivity, examines global and event-level contributions, tests importance stability across source types and time, and locates the model's largest errors. SHAP explains model behavior, not economic causality.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import shap
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipelines.ml.features import build_event_feature_dataset
from pipelines.ml.interpretability import analyze_experiment
from pipelines.ml.xgboost_model import run_xgboost_experiment

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)

## Explain the locked test fold

In [ ]:
dataset = build_event_feature_dataset(horizon=30, include_topics=True)
experiment = run_xgboost_experiment(dataset)
report = analyze_experiment(experiment)
display(pd.Series(report.additivity, name='value'))

## Global model drivers

Mean absolute SHAP measures how strongly a feature moved predictions on average. Mean signed SHAP indicates net direction over this test sample.

In [ ]:
top_global = report.global_importance.head(20)
display(top_global)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=top_global, x='mean_abs_shap', y='feature', color='#2563eb', ax=ax)
ax.set(title='Global XGBoost feature importance on the test fold', xlabel='Mean absolute SHAP value', ylabel='')
plt.tight_layout()
plt.show()

In [ ]:
shap.plots.beeswarm(report.explanation, max_display=15, show=False)
plt.title('Feature values and signed prediction effects')
plt.tight_layout()
plt.show()

## Importance stability

In [ ]:
display(report.stability)
top_features = report.global_importance.head(15)['feature']
source_importance = report.group_importance.loc[
    (report.group_importance['group_type'] == 'event_source')
    & report.group_importance['feature'].isin(top_features)
].pivot(index='feature', columns='group_value', values='mean_abs_shap')
source_importance = source_importance.reindex(top_features)
fig, ax = plt.subplots(figsize=(7, 7))
sns.heatmap(source_importance, annot=True, fmt='.4f', cmap='Blues', ax=ax)
ax.set(title='Importance by event source', xlabel='', ylabel='')
plt.tight_layout()
plt.show()

## Where the model fails

In [ ]:
display(report.error_slices.loc[report.error_slices['slice_type'].isin(['overall', 'event_source'])])
ticker_errors = report.error_slices.loc[report.error_slices['slice_type'] == 'ticker'].nlargest(12, 'mae')
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=ticker_errors, x='mae', y='slice_value', color='#dc2626', ax=ax)
ax.set(title='Largest ticker-level test MAE', xlabel='Mean absolute error', ylabel='')
plt.tight_layout()
plt.show()

In [ ]:
worst_events = report.errors.head(15)
display(worst_events)
worst_key = worst_events.iloc[0]['event_key']
test_frame = experiment.split.test.reset_index(drop=True)
worst_position = test_frame.index[test_frame['event_key'] == worst_key][0]
shap.plots.waterfall(report.explanation[worst_position], max_display=15, show=False)
plt.title(f'Largest-error event: {worst_key}')
plt.tight_layout()
plt.show()

In [ ]:
display(report.local_contributions.loc[report.local_contributions['event_key'] == worst_key])

## Interpretation discipline

Stable SHAP rankings mean the model used similar rules across these slices; they do not mean those rules predicted returns well. The untouched metrics and backtest remain the decision criteria. Error slices are diagnostic hypotheses for future data collection, not permission to tune repeatedly against this test period.